In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append('../../wavelet/')
import wavelet_funcs as wf

import ssqueezepy as sq

### Dummy data - cosine

In [145]:
dt = 0.5
Fs = 1/dt
nt = 20480
t = np.arange(nt)*dt
T = nt*dt

fsignal = 0.03
amp = 1
x = amp*np.cos(2*np.pi*fsignal*t)

P = amp**2 / 2
E = P*T
E_signal = np.sum(x**2) * dt

In [146]:
wlt = sq.Wavelet(('gmw', {'beta':5, 'gamma':3}))
cfs, scales = sq.cwt(x, wavelet=wlt, fs=Fs)

In [147]:
wt_energy = (abs(cfs)**2) * scales.reshape((scales.size, 1)) # Multiply by scale to convert L1 to L2 normalization. Not sure this is correct thing to do

### From Paul Adisson "Illustrated Wavelet Handbook":

Total energy can be recovered from the scalogram (`wt_energy`) by integrating over scale $s$ and time $t$:

$$
E = \frac{1}{C_g} \int_{-\infty}^{\infty}dt \int_0^{\infty}\frac{ds}{s^2} |T(s, t)|^2,
$$

where $C_g$ is the admissibility constant.

In [148]:
Cg = sq.adm_cwt(wlt)

In [149]:
e_scaleint = np.trapezoid(wt_energy/scales.reshape((scales.size, 1))**2, x=scales, axis=0)
factor = 1/Cg
e = np.trapezoid(e_scaleint, x=t)*factor

In [150]:
print(f'Total power  (theory): {P}')
print(f'Total energy (theory): {E}')
print(f'Total energy (computed from signal): {E_signal}')
print(f'Total energy (computed from CWT): {e}')



Total power  (theory): 0.5
Total energy (theory): 5120.0
Total energy (computed from signal): 5121.003391932131
Total energy (computed from CWT): 2560.5509262491896


Energy from CWT is ~2 times smaller than computed from signal, no matter values of beta and gamma. The difference grows with number of data points

### Energy is not preserved in numerical implementation of CWT (https://se.mathworks.com/matlabcentral/answers/339748-cwt-m-normalization#answer_266626)

### Dummy data - white noise

In [158]:
dt = 0.5
Fs = 1/dt
nt = 2048
t = np.arange(nt)*dt
T = nt*dt

sigma = 1
x = np.random.randn(nt) * sigma

P = sigma**2
E = P*T
E_signal = np.sum(x**2) * dt

In [159]:
wlt = sq.Wavelet(('gmw', {'beta':5, 'gamma':3}))
cfs, scales = sq.cwt(x, wavelet=wlt, fs=Fs)

In [160]:
wt_energy = (abs(cfs)**2) * scales.reshape((scales.size, 1)) # Multiply by scale to convert L1 to L2 normalization. Not sure this is correct thing to do

In [161]:
Cg = sq.adm_cwt(wlt)

In [162]:
e_scaleint = np.trapezoid(wt_energy/scales.reshape((scales.size, 1))**2, x=scales, axis=0)
factor = 1/Cg
e = np.trapezoid(e_scaleint, x=t)*factor

In [163]:
print(f'Total power  (theory): {P}')
print(f'Total energy (theory): {E}')
print(f'Total energy (computed from signal): {E_signal}')
print(f'Total energy (computed from CWT): {e}')



Total power  (theory): 1
Total energy (theory): 1024.0
Total energy (computed from signal): 1123.424965799506
Total energy (computed from CWT): 556.4991408993876
